# 9 — source-native / L2 ingestion index and final status matrix

Compact read-only index for the split N4 ingestion notebooks. It links accepted detailed notebooks and promotion/validation reports instead of duplicating every ingestion detail in one monster notebook.

Scope: N4A source-native interactions, N4B L2 biological nodes/context, N4C pharmacology/context/metadata, non-ReMap canonical edge/evidence promotion, ReMap stopped/deferred/support-only status, and ReMap-independent node feature-layer promotions/deferrals.

No canonical KG promotion is performed by this notebook.

## Safety contract

Default execution is safe/read-only: no downloads, no GCS writes, no canonical KG writes, no local Parquet materialization or heavy scans, no shell commands. The notebook only constructs static status matrices from accepted task handoffs and local documentation paths.

In [1]:
from __future__ import annotations
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

READ_ONLY = True
ALLOW_CANONICAL_WRITES = False
ALLOW_GCS_WRITES = False
ALLOW_DOWNLOADS = False
assert READ_ONLY and not ALLOW_CANONICAL_WRITES and not ALLOW_GCS_WRITES and not ALLOW_DOWNLOADS
pd.set_option("display.max_colwidth", 160)

NOTEBOOK_PATHS = {
    "N4A": "notebooks/9a_source_native_interactions_summary.ipynb",
    "N4B": "notebooks/9b_l2_biological_nodes_context_summary.ipynb",
    "N4C": "notebooks/9c_pharmacology_context_metadata_summary.ipynb",
    "N4D_index": "notebooks/9_source_native_l2_ingestion_index.ipynb",
    "N4E_non_remap_promotion": "notebooks/9d_non_remap_part2_canonical_promotion_summary.ipynb",
    "N12_features": "notebooks/12_node_sequence_text_features_summary.ipynb",
}
for rel_path in [NOTEBOOK_PATHS["N4A"], NOTEBOOK_PATHS["N4B"], NOTEBOOK_PATHS["N4C"]]:
    assert (REPO_ROOT / rel_path).exists(), f"missing required detailed notebook: {rel_path}"
NOTEBOOK_PATHS

{'N4A': 'notebooks/9a_source_native_interactions_summary.ipynb',
 'N4B': 'notebooks/9b_l2_biological_nodes_context_summary.ipynb',
 'N4C': 'notebooks/9c_pharmacology_context_metadata_summary.ipynb',
 'N4D_index': 'notebooks/9_source_native_l2_ingestion_index.ipynb',
 'N4E_non_remap_promotion': 'notebooks/9d_non_remap_part2_canonical_promotion_summary.ipynb',
 'N12_features': 'notebooks/12_node_sequence_text_features_summary.ipynb'}

## Top-level ingestion and promotion matrix

Read this table as the final routing/status index. `canonical_promotion_status` deliberately separates biological edge/evidence KG promotion from feature-layer inclusion.

In [2]:
top_level_matrix = pd.DataFrame([
    {"lane":"N4A source-native interactions","detailed_notebook_or_report":NOTEBOOK_PATHS["N4A"],"primary_cards":"t_93348c93; reviewer t_8fba1fa1","gcs_or_artifact_prefixes":"IntAct bounded GCS staging; BioGRID staging; miRNA staging; ReMap all-peak/stopped + CRM support references","review_status":"accepted read-only notebook","canonical_promotion_status":"No promotion from N4A. BioGRID physical PPI later promoted only via non-ReMap lane; ReMap all-peak not promoted.","blockers_or_deferred":"IntAct bounded/no-node-root; BioGRID PTM schema missing; miRNA schema/naming unresolved; all-peak ReMap stopped/deferred."},
    {"lane":"N4B L2 biological nodes/context","detailed_notebook_or_report":NOTEBOOK_PATHS["N4B"],"primary_cards":"t_f5a48cab; reviewer t_5f3e5217","gcs_or_artifact_prefixes":"lncRNA, RBP/RNA CLIP, expression/coexpression, HPA, Complex Portal, UniProt PTM, gene_paralog staged prefixes","review_status":"accepted read-only notebook","canonical_promotion_status":"No canonical KG promotion authorized/performed from N4B; staged-only or feature/context-only claims.","blockers_or_deferred":"lncRNA schema/license; RBP endpoint mapping; HPA all-ENSP policy; protein_complex schema; PTM schema/PSI-MOD; gene_paralog schema/source-order."},
    {"lane":"N4C pharmacology/context/metadata","detailed_notebook_or_report":NOTEBOOK_PATHS["N4C"],"primary_cards":"t_ea7fed1c; reviewer t_a8f7d052","gcs_or_artifact_prefixes":"pharmacology/protein-native, Sci-Plex candidate/context, metadata/features/source coverage artifacts","review_status":"accepted read-only notebook","canonical_promotion_status":"No canonical KG/GCS promotion authorized/performed from N4C.","blockers_or_deferred":"Sci-Plex response edges rejected/downgraded to candidate context; pharmacology/protein-native accepted as staged/non-canonical pending explicit apply gates."},
    {"lane":"Non-ReMap biological edge/evidence canonical promotion","detailed_notebook_or_report":NOTEBOOK_PATHS["N4E_non_remap_promotion"],"primary_cards":"t_61fabcf3 -> t_ce6e158c -> t_17cfc462","gcs_or_artifact_prefixes":"gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/; gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet; gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet","review_status":"validated, reviewer-approved, canonical write accepted","canonical_promotion_status":"Promoted only BioGRID physical PPI to canonical edge/evidence: 3,550 edges and 12,288 evidence rows.","blockers_or_deferred":"All other non-ReMap tranches deferred/include_as_feature_context/rejected per reviewer matrix; BioGRID PTM not promoted."},
    {"lane":"ReMap all-peak / CRM","detailed_notebook_or_report":f"{NOTEBOOK_PATHS['N4A']} plus CRM support task t_3b8a2c4d","primary_cards":"all-peak stopped/deferred line; bounded CRM support/QA t_3b8a2c4d; reviewer t_281f188d","gcs_or_artifact_prefixes":"historical legacy .omoc/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/ (provenance only); current path should be gs://jouvencekb/kg/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/","review_status":"CRM support/QA pilot accepted; all-peak promotion stopped/deferred","canonical_promotion_status":"Not promoted to canonical. CRM is staged support/QA only with evidence_semantics=crm_aggregated_support, not primary observed_binding.","blockers_or_deferred":"No acceptable all-peak promotion; no tf_regulates_gene; bounded first10k chr1 CRM only; no canonical ReMap write."},
    {"lane":"ReMap-independent node feature layer","detailed_notebook_or_report":NOTEBOOK_PATHS["N12_features"],"primary_cards":"t_b5dd2399, t_76c42ace, t_f9ef6389, t_1646af09, t_df83345a, t_61304e4a, t_e6227487, t_c1feb247; plus follow-ups t_607adb48, t_9dfbede5, t_a34d1874","gcs_or_artifact_prefixes":"gs://jouvencekb/kg/v2/features/","review_status":"official feature layer accepted; N12 updated and reviewed","canonical_promotion_status":"Feature-layer promotion, not biological edge/evidence promotion. Current official feature layer has 12 feature tables including molecule_fingerprint and full protein_textual_summary.","blockers_or_deferred":"gene_sequence/gene_genomic_interval deferred; ReMap-independent; no edges/evidence writes."},
])
top_level_matrix

,lane,detailed_notebook_or_report,primary_cards,gcs_or_artifact_prefixes,review_status,canonical_promotion_status,blockers_or_deferred
0,N4A source-native interactions,notebooks/9a_source_native_interactions_summary.ipynb,t_93348c93; reviewer t_8fba1fa1,IntAct bounded GCS staging; BioGRID staging; miRNA staging; ReMap all-peak/stopped + CRM support references,accepted read-only notebook,No promotion from N4A. BioGRID physical PPI later promoted only via non-ReMap lane; ReMap all-peak not promoted.,IntAct bounded/no-node-root; BioGRID PTM schema missing; miRNA schema/naming unresolved; all-peak ReMap stopped/deferred.
1,N4B L2 biological nodes/context,notebooks/9b_l2_biological_nodes_context_summary.ipynb,t_f5a48cab; reviewer t_5f3e5217,"lncRNA, RBP/RNA CLIP, expression/coexpression, HPA, Complex Portal, UniProt PTM, gene_paralog staged prefixes",accepted read-only notebook,No canonical KG promotion authorized/performed from N4B; staged-only or feature/context-only claims.,lncRNA schema/license; RBP endpoint mapping; HPA all-ENSP policy; protein_complex schema; PTM schema/PSI-MOD; gene_paralog schema/source-order.
2,N4C pharmacology/context/metadata,notebooks/9c_pharmacology_context_metadata_summary.ipynb,t_ea7fed1c; reviewer t_a8f7d052,"pharmacology/protein-native, Sci-Plex candidate/context, metadata/features/source coverage artifacts",accepted read-only notebook,No canonical KG/GCS promotion authorized/performed from N4C.,Sci-Plex response edges rejected/downgraded to candidate context; pharmacology/protein-native accepted as staged/non-canonical pending explicit apply gates.
3,Non-ReMap biological edge/evidence canonical promotion,notebooks/9d_non_remap_part2_canonical_promotion_summary.ipynb,t_61fabcf3 -> t_ce6e158c -> t_17cfc462,gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/; gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet; gs://jouvencekb/kg/v2/e...,"validated, reviewer-approved, canonical write accepted","Promoted only BioGRID physical PPI to canonical edge/evidence: 3,550 edges and 12,288 evidence rows.",All other non-ReMap tranches deferred/include_as_feature_context/rejected per reviewer matrix; BioGRID PTM not promoted.
4,ReMap all-peak / CRM,notebooks/9a_source_native_interactions_summary.ipynb plus CRM support task t_3b8a2c4d,all-peak stopped/deferred line; bounded CRM support/QA t_3b8a2c4d; reviewer t_281f188d,.omoc/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/,CRM support/QA pilot accepted; all-peak promotion stopped/deferred,"Not promoted to canonical. CRM is staged support/QA only with evidence_semantics=crm_aggregated_support, not primary observed_binding.",No acceptable all-peak promotion; no tf_regulates_gene; bounded first10k chr1 CRM only; no canonical ReMap write.
5,ReMap-independent node feature layer,notebooks/12_node_sequence_text_features_summary.ipynb,"t_b5dd2399, t_76c42ace, t_f9ef6389, t_1646af09, t_df83345a, t_61304e4a, t_e6227487, t_c1feb247; plus follow-ups t_607adb48, t_9dfbede5, t_a34d1874",gs://jouvencekb/kg/v2/features/,official feature layer accepted; N12 updated and reviewed,"Feature-layer promotion, not biological edge/evidence promotion. Current official feature layer has 12 feature tables including molecule_fingerprint and ful...",gene_sequence/gene_genomic_interval deferred; ReMap-independent; no edges/evidence writes.


## Detailed tranche status matrix

Compact per-tranche index with links/paths to detailed notebooks, reports, GCS prefixes, review status, canonical-promotion status, and blockers/deferred items.

In [3]:
tranche_status = pd.DataFrame([
    {"group":"N4A interactions","tranche":"IntAct corrected/bounded PPI","relation_or_feature":"protein_interacts_protein","detail":NOTEBOOK_PATHS["N4A"],"cards":"t_100231b1; review t_0964be36","staged_or_official_path":"gs://jouvencekb/kg/staging/source-native-expansion/intact-protein-interactions-policy-fixed/runs/20260622T122314Z-bounded100k/","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"bounded sample; no node-root endpoint anti-join; accepted for recovery staging only"},
    {"group":"N4A interactions","tranche":"BioGRID physical PPI","relation_or_feature":"protein_interacts_protein","detail":NOTEBOOK_PATHS["N4E_non_remap_promotion"],"cards":"t_28f83a7b; t_d64b99c0; gate t_ce6e158c; apply t_17cfc462","staged_or_official_path":"gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet + gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet","review_status":"canonical write accepted","canonical_promotion":"promoted edge/evidence","reason_or_caveat":"3,550 edges / 12,288 evidence rows; PTM files from same staging prefix were not promoted"},
    {"group":"N4A interactions","tranche":"BioGRID PTM / ptm_site","relation_or_feature":"ptm_site; protein_has_ptm_site","detail":NOTEBOOK_PATHS["N4A"],"cards":"t_28f83a7b; t_d64b99c0","staged_or_official_path":"gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"ptm_site/protein_has_ptm_site absent from active schema"},
    {"group":"N4A interactions","tranche":"miRNA real-source alias/target path","relation_or_feature":"miRNA aliases/targets","detail":NOTEBOOK_PATHS["N4A"],"cards":"t_f1b51a59; t_1734823c; t_95bbd18c; t_08770b04","staged_or_official_path":"gs://jouvencekb/kg/staging/source-native-expansion/mirna-targets-real/","review_status":"accepted staged path","canonical_promotion":"not promoted","reason_or_caveat":"miRNA node and relation naming/schema unresolved"},
    {"group":"ReMap","tranche":"all-peak ReMap tf_binds_enhancer","relation_or_feature":"tf_binds_enhancer","detail":NOTEBOOK_PATHS["N4A"],"cards":"t_3479936e; t_17f2b3d5; t_8fff8356; t_6d8e46c8; t_5738004a","staged_or_official_path":"all-peak staging lineage only","review_status":"stopped/deferred","canonical_promotion":"not promoted to canonical","reason_or_caveat":"separate ReMap lane; excluded from non-ReMap promotion; no accepted canonical observed_binding promotion"},
    {"group":"ReMap","tranche":"bounded CRM support/QA pilot","relation_or_feature":"tf_binds_enhancer support candidates","detail":"t_3b8a2c4d","cards":"t_3b8a2c4d; review t_281f188d","staged_or_official_path":"historical legacy .omoc/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/ (provenance only); current path should be gs://jouvencekb/kg/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/","review_status":"accepted support/QA pilot","canonical_promotion":"not promoted","reason_or_caveat":"evidence_semantics=crm_aggregated_support; bounded chr1 first10k; no observed_binding or tf_regulates_gene"},
    {"group":"N4B L2 nodes/context","tranche":"lncRNA nodes + LncRNADisease candidates","relation_or_feature":"lncRNA/candidate disease context","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_35dddc93; review t_e3e2a5a0","staged_or_official_path":"gs://jouvencekb/kg/staging/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"license/disease mapping/schema gates"},
    {"group":"N4B L2 nodes/context","tranche":"RBP/RNA CLIP ENCORI/POSTAR context","relation_or_feature":"RNA/protein candidate context","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_89b3ddaf; review t_010bc1e4","staged_or_official_path":"gs://jouvencekb/kg/v2/staging/rbp-rna-clip-encori-pilot-20260622T135045Z/","review_status":"accepted feature-only","canonical_promotion":"not promoted","reason_or_caveat":"no active source-native transcript/protein endpoint relation"},
    {"group":"N4B L2 nodes/context","tranche":"expression/coexpression","relation_or_feature":"feature/context contract","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_c8f1dbc0; review t_a2820a4e","staged_or_official_path":"docs/status only","review_status":"accepted feature/context contract","canonical_promotion":"not promoted","reason_or_caveat":"no active causal/mechanistic relation"},
    {"group":"N4B L2 nodes/context","tranche":"HPA cellular_component/protein localization","relation_or_feature":"protein localization context","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_4bda37e9; t_51714eaf; review t_41852a2b","staged_or_official_path":"gs://jouvencekb/kg/staging/hpa-cellular-components-2026-06-22-rebuild-t_51714eaf/","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"all-ENSP vs canonical-ENSP policy and schema unresolved"},
    {"group":"N4B L2 nodes/context","tranche":"Complex Portal protein_complex","relation_or_feature":"protein_complex membership","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_9d94edb6; review t_c34d9545","staged_or_official_path":"gs://jouvencekb/kg/staging/complex-portal-protein-complexes-2026-06-22/","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"protein_complex node/relation schema and UniProt isoform policy unresolved"},
    {"group":"N4B L2 nodes/context","tranche":"UniProt PTM sites","relation_or_feature":"ptm_site; protein_has_ptm_site","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_ef541e0e; review t_13a70788","staged_or_official_path":"gs://jouvencekb/kg/staging/uniprot-ptm-sites-2026-06-22/","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"ptm_site/protein_has_ptm_site schema and PSI-MOD/isoform handling unresolved"},
    {"group":"N4B L2 nodes/context","tranche":"OpenTargets/Ensembl gene_paralog_gene","relation_or_feature":"gene_paralog_gene","detail":NOTEBOOK_PATHS["N4B"],"cards":"t_e7603b95; review t_b19c67f8","staged_or_official_path":"gs://jouvencekb/kg/staging/source-native-expansion/gene_paralog_gene/opentargets-target-homologues-20260622/","review_status":"accepted staged-only","canonical_promotion":"not promoted","reason_or_caveat":"schema/high-volume/source-order gate unresolved"},
    {"group":"N4C pharmacology/context/metadata","tranche":"pharmacology/protein-native batches","relation_or_feature":"clinical/safety + protein-native relations","detail":NOTEBOOK_PATHS["N4C"],"cards":"t_19516b59; review t_ee55140a","staged_or_official_path":"staged batch reports","review_status":"accepted staged/non-canonical","canonical_promotion":"not promoted","reason_or_caveat":"separate explicit apply/promotion gate required; source-control hygiene caveat"},
    {"group":"N4C pharmacology/context/metadata","tranche":"Sci-Plex candidate/context downgrade","relation_or_feature":"cell_type_responds_to_molecule candidate context","detail":NOTEBOOK_PATHS["N4C"],"cards":"t_63ca49a0; review t_587ab15a","staged_or_official_path":"gs://jouvencekb/kg/staging/cell_type_molecule_candidate_context_sciplex2_20260622_t_63ca49a0","review_status":"accepted candidate/context-only","canonical_promotion":"not promoted","reason_or_caveat":"0 accepted edges/evidence; source lacks response/effect metric vs control"},
    {"group":"N4C pharmacology/context/metadata","tranche":"metadata/features/source coverage","relation_or_feature":"paper/dataset provenance; source coverage","detail":NOTEBOOK_PATHS["N4C"],"cards":"t_67465eac; review t_4de0dfbc","staged_or_official_path":"staged/review artifacts","review_status":"accepted staged artifacts","canonical_promotion":"not promoted by N4C","reason_or_caveat":"not canonical edge promotion material from this gate"},
])
tranche_status

,group,tranche,relation_or_feature,detail,cards,staged_or_official_path,review_status,canonical_promotion,reason_or_caveat
0,N4A interactions,IntAct corrected/bounded PPI,protein_interacts_protein,notebooks/9a_source_native_interactions_summary.ipynb,t_100231b1; review t_0964be36,gs://jouvencekb/kg/staging/source-native-expansion/intact-protein-interactions-policy-fixed/runs/20260622T122314Z-bounded100k/,accepted staged-only,not promoted,bounded sample; no node-root endpoint anti-join; accepted for recovery staging only
1,N4A interactions,BioGRID physical PPI,protein_interacts_protein,notebooks/9d_non_remap_part2_canonical_promotion_summary.ipynb,t_28f83a7b; t_d64b99c0; gate t_ce6e158c; apply t_17cfc462,gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet + gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet,canonical write accepted,promoted edge/evidence,"3,550 edges / 12,288 evidence rows; PTM files from same staging prefix were not promoted"
2,N4A interactions,BioGRID PTM / ptm_site,ptm_site; protein_has_ptm_site,notebooks/9a_source_native_interactions_summary.ipynb,t_28f83a7b; t_d64b99c0,gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/,accepted staged-only,not promoted,ptm_site/protein_has_ptm_site absent from active schema
3,N4A interactions,miRNA real-source alias/target path,miRNA aliases/targets,notebooks/9a_source_native_interactions_summary.ipynb,t_f1b51a59; t_1734823c; t_95bbd18c; t_08770b04,gs://jouvencekb/kg/staging/source-native-expansion/mirna-targets-real/,accepted staged path,not promoted,miRNA node and relation naming/schema unresolved
4,ReMap,all-peak ReMap tf_binds_enhancer,tf_binds_enhancer,notebooks/9a_source_native_interactions_summary.ipynb,t_3479936e; t_17f2b3d5; t_8fff8356; t_6d8e46c8; t_5738004a,all-peak staging lineage only,stopped/deferred,not promoted to canonical,separate ReMap lane; excluded from non-ReMap promotion; no accepted canonical observed_binding promotion
5,ReMap,bounded CRM support/QA pilot,tf_binds_enhancer support candidates,t_3b8a2c4d,t_3b8a2c4d; review t_281f188d,.omoc/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/,accepted support/QA pilot,not promoted,evidence_semantics=crm_aggregated_support; bounded chr1 first10k; no observed_binding or tf_regulates_gene
6,N4B L2 nodes/context,lncRNA nodes + LncRNADisease candidates,lncRNA/candidate disease context,notebooks/9b_l2_biological_nodes_context_summary.ipynb,t_35dddc93; review t_e3e2a5a0,gs://jouvencekb/kg/staging/source-native-expansion/lncrna-l2f-gencode-lncrnadisease-20260622,accepted staged-only,not promoted,license/disease mapping/schema gates
7,N4B L2 nodes/context,RBP/RNA CLIP ENCORI/POSTAR context,RNA/protein candidate context,notebooks/9b_l2_biological_nodes_context_summary.ipynb,t_89b3ddaf; review t_010bc1e4,gs://jouvencekb/kg/v2/staging/rbp-rna-clip-encori-pilot-20260622T135045Z/,accepted feature-only,not promoted,no active source-native transcript/protein endpoint relation
8,N4B L2 nodes/context,expression/coexpression,feature/context contract,notebooks/9b_l2_biological_nodes_context_summary.ipynb,t_c8f1dbc0; review t_a2820a4e,docs/status only,accepted feature/context contract,not promoted,no active causal/mechanistic relation
9,N4B L2 nodes/context,HPA cellular_component/protein localization,protein localization context,notebooks/9b_l2_biological_nodes_context_summary.ipynb,t_4bda37e9; t_51714eaf; review t_41852a2b,gs://jouvencekb/kg/staging/hpa-cellular-components-2026-06-22-rebuild-t_51714eaf/,accepted staged-only,not promoted,all-ENSP vs canonical-ENSP policy and schema unresolved


## Biological edge/evidence canonical promotion lane

Only one N4 biological edge/evidence tranche was promoted to canonical KG from the non-ReMap path. Everything else stayed staged, context-only, or deferred.

In [4]:
edge_promotion_matrix = pd.DataFrame([
    {"promotion_lane":"non-ReMap Part 2 canonical KG","validation_card":"t_61fabcf3","review_card":"t_ce6e158c","apply_card":"t_17cfc462","approved_scope":"BioGRID physical PPI only","canonical_paths":"gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet; gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet","counts":"3,550 edges; 12,288 evidence rows; duplicate/orphan checks 0","status":"promoted and reviewer-accepted"},
    {"promotion_lane":"non-ReMap exclusions","validation_card":"t_61fabcf3","review_card":"t_ce6e158c","apply_card":"none","approved_scope":"IntAct, BioGRID PTM, miRNA, lncRNA, RBP/RNA, expression/coexpression, HPA, Complex Portal, UniProt PTM, gene_paralog, pharmacology, Sci-Plex, metadata/features","canonical_paths":"none","counts":"n/a","status":"not promoted / deferred / include_as_feature_context only"},
    {"promotion_lane":"ReMap tf_binds_enhancer","validation_card":"all-peak stopped/deferred; CRM support t_3b8a2c4d","review_card":"t_281f188d for CRM support/QA","apply_card":"none","approved_scope":"bounded chr1 first10k CRM support/QA only","canonical_paths":"none","counts":"CRM pilot: 10,000 intervals; 338,407 compact support summary rows; support-row count 138,048,185 not materialized as canonical edges","status":"not promoted to canonical; support/QA only"},
])
edge_promotion_matrix

,promotion_lane,validation_card,review_card,apply_card,approved_scope,canonical_paths,counts,status
0,non-ReMap Part 2 canonical KG,t_61fabcf3,t_ce6e158c,t_17cfc462,BioGRID physical PPI only,gs://jouvencekb/kg/v2/edges/protein_interacts_protein.parquet; gs://jouvencekb/kg/v2/evidence/protein_interacts_protein.parquet,"3,550 edges; 12,288 evidence rows; duplicate/orphan checks 0",promoted and reviewer-accepted
1,non-ReMap exclusions,t_61fabcf3,t_ce6e158c,none,"IntAct, BioGRID PTM, miRNA, lncRNA, RBP/RNA, expression/coexpression, HPA, Complex Portal, UniProt PTM, gene_paralog, pharmacology, Sci-Plex, metadata/features",none,n/a,not promoted / deferred / include_as_feature_context only
2,ReMap tf_binds_enhancer,all-peak stopped/deferred; CRM support t_3b8a2c4d,t_281f188d for CRM support/QA,none,bounded chr1 first10k CRM support/QA only,none,"CRM pilot: 10,000 intervals; 338,407 compact support summary rows; support-row count 138,048,185 not materialized as canonical edges",not promoted to canonical; support/QA only


## Feature-layer promotions are separate from biological edge/evidence promotion

The ReMap-independent feature wave writes model feature tables under `features/`. These are not biological KG edges and do not create evidence assertions.

In [5]:
feature_layer_matrix = pd.DataFrame([
    {"feature_table":"cell_line_textual_summary.parquet","source":"Cellosaurus","official_path":"gs://jouvencekb/kg/v2/features/cell_line_textual_summary.parquet","rows_or_status":"1,140","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"cell_type_textual_summary.parquet","source":"Cell Ontology","official_path":"gs://jouvencekb/kg/v2/features/cell_type_textual_summary.parquet","rows_or_status":"3,135","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"disease_textual_summary.parquet","source":"OpenTargets","official_path":"gs://jouvencekb/kg/v2/features/disease_textual_summary.parquet","rows_or_status":"26,395","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"gene_textual_summary.parquet","source":"OpenTargets","official_path":"gs://jouvencekb/kg/v2/features/gene_textual_summary.parquet","rows_or_status":"212,029","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"molecule_textual_summary.parquet","source":"ChEMBL","official_path":"gs://jouvencekb/kg/v2/features/molecule_textual_summary.parquet","rows_or_status":"22,230","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"pathway_textual_summary.parquet","source":"GO","official_path":"gs://jouvencekb/kg/v2/features/pathway_textual_summary.parquet","rows_or_status":"37,492","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"phenotype_textual_summary.parquet","source":"HPO","official_path":"gs://jouvencekb/kg/v2/features/phenotype_textual_summary.parquet","rows_or_status":"13,810","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"protein_sequence.parquet","source":"Ensembl release 114 / GRCh38","official_path":"gs://jouvencekb/kg/v2/features/protein_sequence.parquet","rows_or_status":"112,051","feature_status":"official feature layer","cards":"t_f9ef6389 -> t_61304e4a -> t_ed43b234"},
    {"feature_table":"tissue_textual_summary.parquet","source":"UBERON","official_path":"gs://jouvencekb/kg/v2/features/tissue_textual_summary.parquet","rows_or_status":"11,942","feature_status":"official feature layer","cards":"t_61304e4a -> t_ed43b234"},
    {"feature_table":"transcript_sequence.parquet","source":"Ensembl release 114 / GRCh38","official_path":"gs://jouvencekb/kg/v2/features/transcript_sequence.parquet","rows_or_status":"187,268","feature_status":"official feature layer","cards":"t_f9ef6389 -> t_61304e4a -> t_ed43b234"},
    {"feature_table":"molecule_fingerprint.parquet","source":"nodes/molecule.parquet.smiles + RDKit Morgan binary radius=2 n_bits=2048","official_path":"gs://jouvencekb/kg/v2/features/molecule_fingerprint.parquet","rows_or_status":"18,614","feature_status":"official feature layer","cards":"t_0cc11b0f -> t_3c43f536 -> t_607adb48"},
    {"feature_table":"protein_textual_summary.parquet","source":"UniProtKB accepted comment blocks; CC BY 4.0","official_path":"gs://jouvencekb/kg/v2/features/protein_textual_summary.parquet","rows_or_status":"162,163 / 233,995 proteins (69.301908%)","feature_status":"official feature layer","cards":"documented by t_9dfbede5/t_a34d1874; report docs/protein_textual_summary_official_promotion_report.md"},
    {"feature_table":"gene_sequence.parquet","source":"not selected","official_path":"none","rows_or_status":"not staged","feature_status":"deferred / not_promoted","cards":"t_0cc11b0f; t_3c43f536; t_9dfbede5"},
    {"feature_table":"gene_genomic_interval.parquet","source":"not selected","official_path":"none","rows_or_status":"not staged","feature_status":"deferred / not_promoted","cards":"t_0cc11b0f; t_3c43f536; t_9dfbede5"},
])
feature_layer_matrix

,feature_table,source,official_path,rows_or_status,feature_status,cards
0,cell_line_textual_summary.parquet,Cellosaurus,gs://jouvencekb/kg/v2/features/cell_line_textual_summary.parquet,"1,140",official feature layer,t_61304e4a -> t_ed43b234
1,cell_type_textual_summary.parquet,Cell Ontology,gs://jouvencekb/kg/v2/features/cell_type_textual_summary.parquet,"3,135",official feature layer,t_61304e4a -> t_ed43b234
2,disease_textual_summary.parquet,OpenTargets,gs://jouvencekb/kg/v2/features/disease_textual_summary.parquet,"26,395",official feature layer,t_61304e4a -> t_ed43b234
3,gene_textual_summary.parquet,OpenTargets,gs://jouvencekb/kg/v2/features/gene_textual_summary.parquet,"212,029",official feature layer,t_61304e4a -> t_ed43b234
4,molecule_textual_summary.parquet,ChEMBL,gs://jouvencekb/kg/v2/features/molecule_textual_summary.parquet,"22,230",official feature layer,t_61304e4a -> t_ed43b234
5,pathway_textual_summary.parquet,GO,gs://jouvencekb/kg/v2/features/pathway_textual_summary.parquet,"37,492",official feature layer,t_61304e4a -> t_ed43b234
6,phenotype_textual_summary.parquet,HPO,gs://jouvencekb/kg/v2/features/phenotype_textual_summary.parquet,"13,810",official feature layer,t_61304e4a -> t_ed43b234
7,protein_sequence.parquet,Ensembl release 114 / GRCh38,gs://jouvencekb/kg/v2/features/protein_sequence.parquet,"112,051",official feature layer,t_f9ef6389 -> t_61304e4a -> t_ed43b234
8,tissue_textual_summary.parquet,UBERON,gs://jouvencekb/kg/v2/features/tissue_textual_summary.parquet,"11,942",official feature layer,t_61304e4a -> t_ed43b234
9,transcript_sequence.parquet,Ensembl release 114 / GRCh38,gs://jouvencekb/kg/v2/features/transcript_sequence.parquet,"187,268",official feature layer,t_f9ef6389 -> t_61304e4a -> t_ed43b234


## Explicit not-promoted / deferred sections

These rows are intentionally explicit because no separate promotion card actually ran for them. They should not be inferred as canonical just because staged artifacts or accepted notebooks exist.

In [6]:
not_promoted_or_deferred = pd.DataFrame([
    {"category":"ReMap","item":"all-peak ReMap tf_binds_enhancer","status":"deferred / stopped / not_promoted","why":"separate ReMap lane; no accepted all-peak canonical observed_binding promotion"},
    {"category":"ReMap","item":"bounded CRM tf_binds_enhancer support","status":"staged support/QA only / not_promoted","why":"crm_aggregated_support semantics; no primary observed_binding; no canonical writes"},
    {"category":"non-ReMap KG","item":"IntAct bounded PPI","status":"not_promoted","why":"bounded/no-node-root caveat"},
    {"category":"non-ReMap KG","item":"BioGRID PTM / ptm_site","status":"not_promoted","why":"schema absent"},
    {"category":"non-ReMap KG","item":"miRNA target path","status":"not_promoted","why":"node/relation schema/naming unresolved"},
    {"category":"L2 nodes/context","item":"lncRNA/RBP/expression/HPA/Complex Portal/PTM/gene_paralog","status":"staged-only or feature/context-only / not_promoted","why":"each has unresolved schema, endpoint, license, or policy gate"},
    {"category":"N4C","item":"Sci-Plex cell_type_responds_to_molecule","status":"candidate/context-only / not_promoted","why":"0 accepted response edges/evidence; source lacks response/effect metric"},
    {"category":"feature layer","item":"gene_sequence / gene_genomic_interval","status":"deferred / not_promoted","why":"no reviewed coordinate/reference-build/mapping/strand/length/FASTA checksum policy"},
])
not_promoted_or_deferred

,category,item,status,why
0,ReMap,all-peak ReMap tf_binds_enhancer,deferred / stopped / not_promoted,separate ReMap lane; no accepted all-peak canonical observed_binding promotion
1,ReMap,bounded CRM tf_binds_enhancer support,staged support/QA only / not_promoted,crm_aggregated_support semantics; no primary observed_binding; no canonical writes
2,non-ReMap KG,IntAct bounded PPI,not_promoted,bounded/no-node-root caveat
3,non-ReMap KG,BioGRID PTM / ptm_site,not_promoted,schema absent
4,non-ReMap KG,miRNA target path,not_promoted,node/relation schema/naming unresolved
5,L2 nodes/context,lncRNA/RBP/expression/HPA/Complex Portal/PTM/gene_paralog,staged-only or feature/context-only / not_promoted,"each has unresolved schema, endpoint, license, or policy gate"
6,N4C,Sci-Plex cell_type_responds_to_molecule,candidate/context-only / not_promoted,0 accepted response edges/evidence; source lacks response/effect metric
7,feature layer,gene_sequence / gene_genomic_interval,deferred / not_promoted,no reviewed coordinate/reference-build/mapping/strand/length/FASTA checksum policy


## Validation commands for this index notebook

Recommended lightweight validation from the repository root:

```bash
uv run python - <<'PY'
from pathlib import Path
import nbformat
from nbclient import NotebookClient
p = Path('notebooks/9_source_native_l2_ingestion_index.ipynb')
nb = nbformat.read(p, as_version=4)
nbformat.validate(nb)
NotebookClient(nb, timeout=120, kernel_name='python3').execute()
print('nbformat+nbclient PASS', p)
PY
uv run --group dev pytest tests/test_notebook_structure.py -q
```

Reviewer checklist: read-only posture, links to N4A/N4B/N4C/N4E/N12, separation between edge/evidence KG promotion and feature-layer promotion, explicit `not promoted` / `deferred` rows, and no canonical KG promotion.

In [7]:
assert READ_ONLY
assert not ALLOW_CANONICAL_WRITES
assert "not_promoted" in " ".join(tranche_status["canonical_promotion"].astype(str).tolist() + not_promoted_or_deferred["status"].astype(str).tolist())
assert "BioGRID physical PPI" in edge_promotion_matrix.loc[0, "approved_scope"]
assert feature_layer_matrix["feature_status"].str.contains("official feature layer").sum() >= 12
assert (feature_layer_matrix["feature_table"] == "gene_sequence.parquet").any()
assert feature_layer_matrix.loc[feature_layer_matrix["feature_table"] == "gene_sequence.parquet", "feature_status"].iloc[0] == "deferred / not_promoted"
assert top_level_matrix["lane"].str.contains("ReMap-independent node feature layer", regex=False).any()
"N4D index self-check PASS"

'N4D index self-check PASS'